In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage

# I'm using seaborn for it's fantastic default styles
import seaborn as sns
sns.set_style("whitegrid")

%matplotlib inline
%load_ext autoreload
%autoreload 2

from tutils import BaseStateSystem

In [3]:
def laplacian1D(a, dx):
    return scipy.ndimage.correlate1d(a, weights=[1.0, -2.0, 1.0], mode="wrap") / (dx ** 2)

def laplacian2D(a, dx):
    weights = np.array([[0.0, 1.0, 0.0],
                        [1.0, -4.0, 1.0],
                        [0.0, 1.0, 0.0]])
    return scipy.ndimage.correlate(a, weights=weights, mode="wrap") / (dx ** 2)


In [4]:
class OneDimensionalDiffusionEquation(BaseStateSystem):
    def __init__(self, D):
        self.D = D
        self.width = 1000
        self.dx = 10 / self.width
        self.dt = 0.9 * (self.dx ** 2) / (2 * D)
        self.steps = int(0.1 / self.dt)
        
    def initialise(self):
        self.t = 0
        self.X = np.linspace(-5,5,self.width)
        self.a = np.exp(-self.X**2)
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):      
        La = laplacian1D(self.a, self.dx)
        delta_a = self.dt * (self.D * La)       
        self.a += delta_a
        
    def draw(self, ax):
        ax.clear()
        ax.plot(self.X,self.a, color="r")
        ax.set_ylim(0,1)
        ax.set_xlim(-5,5)
        ax.set_title("t = {:.2f}".format(self.t))
    
one_d_diffusion = OneDimensionalDiffusionEquation(D=1)

one_d_diffusion.plot_time_evolution("diffusion.gif")

In [5]:
class ReactionEquation(BaseStateSystem):
    def __init__(self, Ra, Rb):
        self.Ra = Ra
        self.Rb = Rb
        self.dt = 0.02
        self.steps = int(0.48 / self.dt)
        
    def initialise(self):
        self.t = 0
        self.a = 0.1
        self.b = 0.7
        self.Ya = []
        self.Yb = []
        self.X = []
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):      
        delta_a = self.dt * self.Ra(self.a,self.b)      
        delta_b = self.dt * self.Rb(self.a,self.b)      

        self.a += delta_a
        self.b += delta_b
        
    def draw(self, ax):
        ax.clear()
        
        self.X.append(self.t)
        self.Ya.append(self.a)
        self.Yb.append(self.b)

        ax.plot(self.X,self.Ya, color="r", label="u")
        ax.plot(self.X,self.Yb, color="b", label="v")
        ax.legend()
        
        ax.set_ylim(0,1)
        ax.set_xlim(0,24)
        ax.set_xlabel("t")
        ax.set_ylabel("Concentrations")
        
alpha,beta,gamma =  0.425, 0.075, 0.9

def Ra_Schnakenberg(u,v): return gamma*(alpha-u+u**2*v)
def Rb_Schnakenberg(u,v): return gamma*(beta-u**2*v)
    
one_d_reaction = ReactionEquation(Ra_Schnakenberg, Rb_Schnakenberg)
one_d_reaction.plot_time_evolution("reaction_schnakenberg.gif", n_steps=60)

In [6]:
F,k=0.75/7,0.15-0.75/7
def Ra_gray_scott(u,v): return F*(1-u)-u*v**2
def Rb_gray_scott(u,v): return u*v**2-(F+k)*v
one_d_reaction = ReactionEquation(Ra_gray_scott, Rb_gray_scott)
one_d_reaction.plot_time_evolution("reaction_gray_scott.gif", n_steps=60)

In [ ]:
def random_initialiser_Schnakenberg(shape):
    return(
        np.random.normal(loc=1.0, scale=0.05, size=shape),
        np.random.normal(loc=0.9, scale=0.05, size=shape)
    )

class OneDimensionalRDEquations(BaseStateSystem):
    def __init__(self, Da, Db, Ra, Rb,
                 initialiser=random_initialiser_Schnakenberg,
                 width=1000, dx=1, 
                 dt=0.1, steps=1,y_lim=10):
        
        self.Da = Da
        self.Db = Db
        self.Ra = Ra
        self.Rb = Rb
        
        self.initialiser = initialiser
        self.width = width
        self.dx = dx
        self.dt = dt
        self.steps = steps
        self.y_lim = y_lim
        
    def initialise(self):
        self.t = 0
        self.a, self.b = self.initialiser(self.width)
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):
        
        # unpack so we don't have to keep writing "self"
        a,b,Da,Db,Ra,Rb,dt,dx = (
            self.a, self.b,
            self.Da, self.Db,
            self.Ra, self.Rb,
            self.dt, self.dx
        )
        
        La = laplacian1D(a, dx)
        Lb = laplacian1D(b, dx)
        
        delta_a = dt * (Da * La + Ra(a,b))
        delta_b = dt * (Db * Lb + Rb(a,b))
        
        self.a += delta_a
        self.b += delta_b
        
    def draw(self, ax):
        ax.clear()
        ax.plot(self.a, color="r", label="u")
        ax.plot(self.b, color="b", label="v")
        ax.legend()
        ax.set_ylim(-1,self.y_lim)
        ax.set_title("t = {:.2f}".format(self.t))
        
alpha,beta,gamma=0.1268,0.7924,1
Da = 0.1
Db = 10

width = 50
dx = 0.1
dt = 0.0002

OneDimensionalRDEquations(
    Da, Db, Ra_Schnakenberg, Rb_Schnakenberg,initialiser=random_initialiser_Schnakenberg,
    width=width, dx=dx, dt=dt, 
    steps=2000,y_lim=10
).plot_time_evolution("1dRD_Schnakenberg.gif", n_steps=150)
# ).display_time_evolution(n_steps=150)

In [8]:
def random_initialiser_GrayScott(shape):
    return(
        np.random.normal(loc=0.754, scale=0.05, size=shape),
        np.random.normal(loc=0.122, scale=0.05, size=shape)
    )
F,k=0.030,0.062
Da = 0.02
Db = 0.01

width = 500
dx = 1
dt = 0.001

OneDimensionalRDEquations(
    Da, Db, Ra_gray_scott, Rb_gray_scott,initialiser=random_initialiser_GrayScott,
    width=width, dx=dx, dt=dt, 
    steps=2000,y_lim=2
).plot_time_evolution("1dRD_Gray-Scott.gif", n_steps=150)
# ).display_time_evolution(n_steps=150)

In [52]:
class TwoDimensionalRDEquations(BaseStateSystem):
    def __init__(self, Da, Db, Ra, Rb,
                 initialiser=random_initialiser_Schnakenberg,
                 width=1000, height=1000,
                 dx=1, dt=0.1, steps=1):
        
        self.Da = Da
        self.Db = Db
        self.Ra = Ra
        self.Rb = Rb

        self.initialiser = initialiser
        self.width = width
        self.height = height
        self.shape = (width, height)
        self.dx = dx
        self.dt = dt
        self.steps = steps
        
    def initialise(self):
        self.t = 0
        self.a, self.b = self.initialiser(self.shape)
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):
        
        # unpack so we don't have to keep writing "self"
        a,b,Da,Db,Ra,Rb,dt,dx = (
            self.a, self.b,
            self.Da, self.Db,
            self.Ra, self.Rb,
            self.dt, self.dx
        )
        
        La = laplacian2D(a, dx)
        Lb = laplacian2D(b, dx)
        
        delta_a = dt * (Da * La + Ra(a,b))
        delta_b = dt * (Db * Lb + Rb(a,b))
        
        self.a += delta_a
        self.b += delta_b
        
    def draw(self, ax):
        ax[0].clear()
        ax[1].clear()

        ax[0].imshow(self.a, cmap='jet')
        ax[1].imshow(self.b, cmap='brg')
        
        ax[0].grid(False)
        ax[1].grid(False)
        
        ax[0].set_title("u, t = {:.2f}".format(self.t))
        ax[1].set_title("v, t = {:.2f}".format(self.t))
        
    def initialise_figure(self):
        fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,6))
        return fig, ax

Da = 1
Db = 20

width = 200
dx = 1
dt = 0.002

TwoDimensionalRDEquations(
    Da, Db, Ra_Schnakenberg, Rb_Schnakenberg,initialiser=random_initialiser_Schnakenberg, 
    width=width, height=width, 
    dx=dx, dt=dt, steps=500
).plot_evolution_outcome("2dRD_Schnakenberg.png", n_steps=150)

In [50]:
def gs_initialiser(shape):
        a = np.ones(shape)
        b = np.zeros(shape)
        centre = int(shape[0] / 2)
        
        a[centre-20:centre+20,centre-20:centre+20] = 0.5
        b[centre-20:centre+20,centre-20:centre+20] = 0.25
        
        a += np.random.normal(scale=0.05, size=shape)
        b += np.random.normal(scale=0.05, size=shape)
        
        return a,b 

# interesting parameters taken from http://www.aliensaint.com/uo/java/rd/
params = [
    [0.16, 0.08, 0.035, 0.065], 
    [0.14, 0.06, 0.035, 0.065], 
    [0.16, 0.08, 0.06, 0.062], 
    [0.19, 0.05, 0.06, 0.062], 
    [0.16, 0.08, 0.02, 0.055], 
    [0.16, 0.08, 0.05, 0.065], 
    [0.16, 0.08, 0.054, 0.063], 
    [0.16, 0.08, 0.035, 0.06]
]

for i, param in enumerate(params):
    
    Da, Db, f, k = param
    
    def Ra(a,b): return - a*b*b + f*(1-a)
    def Rb(a,b): return + a*b*b - (f+k)*b

    width = 200
    dx = 1
    dt = 1

    TwoDimensionalRDEquations(
        Da, Db, Ra, Rb,
        initialiser=gs_initialiser,
        width=width, height=width, 
        dx=dx, dt=dt, steps=200
    ).plot_evolution_outcome("gs_{}.png".format(i), n_steps=100)
